# Task 2: Tell-the-Time Network

We predict the time shown on an analog clock from a grayscale image (labels: integer hour and minute).

**Plan**
1. Load the data (80 / 10 / 10 split) and define the *common-sense error* (circular time difference in minutes).
2. One CNN class (`ClockNet`) with four interchangeable output formulations: classification, regression, multi-head and trigonometric encoding.
3. Compare the formulations on the 75x75 images, using the **validation** set only.
4. Tune the backbone (learning rate, width, dropout) for the best formulation.
5. Train all formulations on the 150x150 images with the tuned backbone and report the **test** error (the test set is only touched here).

In [ ]:
import os
import glob
import math
import time
import random
import zipfile
import itertools
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display   # keeps display() working when the notebook is exported to a .py script

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

In [ ]:
# Hardware device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed = 67):
    """Seed every random number generator we use (67 = our group number, as in Task 1)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

## Data

The data are downloaded on demand into the same `./data` folder as in Task 1 (nothing is downloaded if the files are already there).
Images are scaled to [0, 1] and normalised with the mean / std of the training images. We do **not** use horizontal flips as augmentation, because a mirrored clock shows a different time.

In [ ]:
DATA_URLS = {
    75: "https://surfdrive.surf.nl/public.php/dav/files/Nznt5c48Mzlb2HY/A1_data_75.zip",
    150: "https://surfdrive.surf.nl/public.php/dav/files/Nznt5c48Mzlb2HY/A1_data_150.zip",
}

def find_file(folder, keyword):
    """Find e.g. images.npy anywhere below `folder` (the zip file may contain a sub-folder)."""
    matches = glob.glob(os.path.join(folder, "**", f"*{keyword}*.npy"), recursive = True)
    return matches[0] if matches else None

def download_if_missing(size):
    """Download + unzip the dataset with the given image size, unless it is already in ./data."""
    folder = f"./data/A1_data_{size}"
    if find_file(folder, "images") is None:
        print(f"Downloading the {size}x{size} dataset ...")
        os.makedirs(folder, exist_ok = True)
        zip_path = folder + ".zip"
        urlretrieve(DATA_URLS[size], zip_path)
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(folder)
        os.remove(zip_path)
    return folder

def load_clock_data(size, seed = 67):
    """Load the clock images of one size and return the train / validation / test datasets (80 / 10 / 10)."""
    folder = download_if_missing(size)
    images = np.load(find_file(folder, "images"))
    labels = np.load(find_file(folder, "labels"))
    print(f"{size}x{size} data: images {images.shape}, labels {labels.shape}")

    images = images.astype(np.float32)
    if images.max() > 1.0:                          # pixel values 0-255 -> 0-1
        images /= 255.0
    # (N, H, W) -> (N, 1, H, W): add the channel dimension that nn.Conv2d expects
    images = torch.from_numpy(images).reshape(-1, 1, images.shape[1], images.shape[2])
    labels = torch.from_numpy(labels.astype(np.int64))          # (N, 2): hour, minute

    # Seeded random split. random_split shuffles the indices, so the order of the raw files does not matter.
    dataset = TensorDataset(images, labels)
    n_train = int(0.8 * len(dataset))
    n_validation = int(0.1 * len(dataset))
    n_test = len(dataset) - n_train - n_validation
    train, validation, test = random_split(dataset, [n_train, n_validation, n_test],
                                           generator = torch.Generator().manual_seed(seed))

    # Normalise with the mean / std of (2000 of) the TRAINING images only.
    # In place, so the dataset (which shares this memory) sees the normalised images.
    reference = images[train.indices[:2000]]
    images.sub_(reference.mean()).div_(reference.std())
    return train, validation, test

In [ ]:
clock_train, clock_validation, clock_test = load_clock_data(75)    # prototype on the 75x75 images
print(f"Train / validation / test: {len(clock_train)} / {len(clock_validation)} / {len(clock_test)}")

# Check what the labels look like (is the hour 0-11 or 1-12?)
all_labels = clock_train.dataset.tensors[1]
print("Hours in the labels:", sorted(all_labels[:, 0].unique().tolist()))
print("Minutes in the labels:", all_labels[:, 1].min().item(), "to", all_labels[:, 1].max().item())

In [ ]:
# Visualise (hour, minute)
fig, ax = plt.subplots(1, 3, figsize = (9, 3))
for i in range(3):
    image, (hour, minute) = clock_train[i]
    ax[i].imshow(image.squeeze(), cmap = "gray")
    ax[i].set_title(f"({hour.item()}, {minute.item()})")
    ax[i].axis("off")
plt.tight_layout()
plt.show()

## Evaluation metric: common-sense error

Every time is converted to *minutes since 12:00* (0 to 719). The error is the shorter way around the 12-hour clock, so 11:55 vs 0:05 is 10 minutes, not 11 h 50 min.

In [ ]:
def time_in_minutes(labels):
    """Convert (hour, minute) labels into minutes since 12:00, in [0, 720). Hour 12 and hour 0 are the same."""
    return (labels[:, 0] % 12) * 60 + labels[:, 1]

def common_sense_error(pred_minutes, true_minutes):
    """Absolute time difference in minutes, taking the 12-hour wrap-around into account."""
    diff = torch.remainder(pred_minutes - true_minutes, 720)     # in [0, 720), also for negative predictions
    return torch.minimum(diff, 720 - diff)                        # the shorter way around the clock

# Check with the example from the assignment: predicted 11:55, target 0:05 -> 10 minutes
example = common_sense_error(torch.tensor([11 * 60 + 55.0]), torch.tensor([0 * 60 + 5.0]))
print(example.item())
assert example.item() == 10.0

## Model building

One CNN class with a **shared backbone** (convolutional blocks + fully connected layers) and an output part that depends on `output_type`:

| `output_type` | Network output | Loss | Prediction (in minutes) |
|---|---|---|---|
| `classification` | `n_classes` logits (24 = 30-minute bins ... 720 = 1-minute bins) | cross-entropy | centre of the predicted bin |
| `regression` | 1 value = time as a fraction of 12 hours, in [0, 1) | MSE / L1 / SmoothL1 | value x 720 |
| `multihead` | `out_hour` (12 logits) and `out_minute` (60 logits or 1 value) | CE(hour) + alpha x loss(minute) | 60 x hour + minute |
| `trig` | (sin, cos) of the hour-hand angle and of the minute-hand angle | MSE / L1 | `atan2` of both angles, hour chosen to fit the minute |

In [ ]:
class ClockNet(nn.Module):
    def __init__(self,
            output_type = "regression",     # "classification", "regression", "multihead" or "trig"
            n_classes = 24,                 # classification only: 24 -> 30-minute bins, ..., 720 -> 1-minute bins
            minute_mode = "class",          # multihead only: minute head with 60 classes ("class") or one value ("reg")
            cnn_channels = [32, 64, 128, 256],   # output channels of each convolutional block
            kernel_size = 3,                # padding = kernel_size // 2 keeps the image size, so use odd sizes
            batch_norm = True,
            hidden_dims = [256],            # fully connected layers shared by all heads
            activation = nn.ReLU,
            dropout = 0.0):
        super().__init__()
        self.output_type = output_type

        #---------------Shared convolutional backbone---------------
        layers = []
        in_ch = 1                           # grayscale images
        for out_ch in cnn_channels:
            layers.append(nn.Conv2d(in_ch, out_ch, kernel_size, padding = kernel_size // 2))
            if batch_norm:
                layers.append(nn.BatchNorm2d(out_ch))
            layers.append(activation())
            layers.append(nn.MaxPool2d(2))
            in_ch = out_ch
        self.features = nn.Sequential(*layers)

        # Fixed 4x4 feature map before the linear layers, so the same network accepts 75x75 and 150x150 images.
        # We keep a 4x4 map (instead of one average per channel) because the POSITION of the hands matters.
        self.pool = nn.AdaptiveAvgPool2d(4)

        #---------------Shared fully connected layers---------------
        fc_layers = [nn.Flatten()]
        input_size = cnn_channels[-1] * 4 * 4
        for hidden_size in hidden_dims:
            fc_layers.append(nn.Linear(input_size, hidden_size))
            fc_layers.append(activation())
            if dropout > 0:
                fc_layers.append(nn.Dropout(dropout))
            input_size = hidden_size
        self.fc = nn.Sequential(*fc_layers)

        #---------------Output head(s): no activation, they output logits / raw values---------------
        if output_type == "classification":
            self.head = nn.Linear(input_size, n_classes)
        elif output_type == "regression":
            self.head = nn.Linear(input_size, 1)
        elif output_type == "trig":
            self.head = nn.Linear(input_size, 4)        # sin / cos of the hour angle and of the minute angle
        elif output_type == "multihead":
            self.hour_head = nn.Linear(input_size, 12)
            self.minute_head = nn.Linear(input_size, 60 if minute_mode == "class" else 1)
        else:
            raise ValueError(f"Unknown output_type: {output_type!r}")

    def forward(self, x):
        z = self.fc(self.pool(self.features(x)))        # representation shared by all heads
        if self.output_type == "multihead":
            return self.hour_head(z), self.minute_head(z)       # out_hour, out_minute = model(x)
        out = self.head(z)
        if self.output_type == "trig":
            # Scale each (sin, cos) pair to length 1: the prediction is then a point on the unit circle.
            # (No ReLU / sigmoid: sin and cos can be negative.)
            out = torch.cat([F.normalize(out[:, :2], dim = 1), F.normalize(out[:, 2:], dim = 1)], dim = 1)
        return out

### Targets, losses and decoding

For every formulation we need (1) the **target** that is built from the (hour, minute) label, (2) the **loss**, and (3) a way to turn the network output back into a time in minutes, so that all formulations are scored with the same common-sense error.

In [ ]:
# Losses for outputs that are real numbers (regression, minute head, trigonometric encoding).
# SmoothL1 uses beta = 0.05 instead of the default 1.0: our targets are in [0, 1], so with beta = 1
# SmoothL1 would simply be 0.5 * MSE.
REG_LOSSES = {"mse": nn.MSELoss(), "l1": nn.L1Loss(), "smoothl1": nn.SmoothL1Loss(beta = 0.05)}

def trig_encode(labels):
    """(hour, minute) -> (sin, cos) of the hour-hand angle and of the minute-hand angle, shape (B, 4)."""
    angle_h = 2 * math.pi * time_in_minutes(labels).float() / 720       # = 2 pi (hour + minute / 60) / 12
    angle_m = 2 * math.pi * labels[:, 1].float() / 60
    return torch.stack([torch.sin(angle_h), torch.cos(angle_h), torch.sin(angle_m), torch.cos(angle_m)], dim = 1)

def compute_loss(outputs, labels, params):
    kind = params["output_type"]
    t = time_in_minutes(labels).float()                  # true time in minutes since 12:00
    hour, minute = labels[:, 0] % 12, labels[:, 1]

    if kind == "classification":
        bin_size = 720 // params["n_classes"]
        return F.cross_entropy(outputs, (t // bin_size).long())          # class = index of the time bin

    if kind == "regression":
        return REG_LOSSES[params["loss"]](outputs.squeeze(1), t / 720)   # target: fraction of 12 hours, in [0, 1)

    if kind == "multihead":
        out_hour, out_minute = outputs
        loss_hour = F.cross_entropy(out_hour, hour)
        if params["minute_mode"] == "class":
            loss_minute = F.cross_entropy(out_minute, minute)
        else:
            loss_minute = REG_LOSSES[params["loss"]](out_minute.squeeze(1), minute / 60)
        return loss_hour + params["alpha"] * loss_minute                 # L = L_hour + alpha * L_minute

    if kind == "trig":
        return REG_LOSSES[params["loss"]](outputs, trig_encode(labels))

def outputs_to_minutes(outputs, params):
    """Turn network outputs into a predicted time in minutes since 12:00 (the wrap-around is handled by common_sense_error)."""
    kind = params["output_type"]

    if kind == "classification":
        bin_size = 720 // params["n_classes"]
        return outputs.argmax(1).float() * bin_size + (bin_size - 1) / 2      # centre of the predicted bin

    if kind == "regression":
        return outputs.squeeze(1) * 720

    if kind == "multihead":
        out_hour, out_minute = outputs
        if params["minute_mode"] == "class":
            minute = out_minute.argmax(1).float()
        else:
            minute = out_minute.squeeze(1) * 60
        return out_hour.argmax(1).float() * 60 + minute

    if kind == "trig":
        angle_h = torch.atan2(outputs[:, 0], outputs[:, 1]) % (2 * math.pi)     # hour-hand angle in [0, 2 pi)
        angle_m = torch.atan2(outputs[:, 2], outputs[:, 3]) % (2 * math.pi)     # minute-hand angle
        rough = angle_h / (2 * math.pi) * 720          # rough time from the hour hand
        minute = angle_m / (2 * math.pi) * 60          # exact minute from the minute hand
        # The hour hand only gives a rough time, the minute hand the exact minute.
        # Pick the hour h (0..11) for which 60 * h + minute is closest to the rough time.
        hours = torch.arange(12, device = outputs.device).float().view(1, 12)
        candidates = hours * 60 + minute.view(-1, 1)                            # (B, 12)
        diff = torch.remainder(candidates - rough.view(-1, 1), 720)
        diff = torch.minimum(diff, 720 - diff)
        return candidates.gather(1, diff.argmin(1, keepdim = True)).squeeze(1)

In [ ]:
# Sanity check: perfect trigonometric outputs must decode back to the true time (error ~ 0)
check = torch.tensor([[0, 5], [11, 55], [3, 0], [2, 34], [7, 59]])      # (hour, minute)
decoded = outputs_to_minutes(trig_encode(check), {"output_type": "trig"})
print(common_sense_error(decoded, time_in_minutes(check).float()))

### Training

`run_experiment` has the same structure as in Task 1: parameters in a dict, DataLoaders, model, optimizer, training / validation loop, restore of the best weights, test evaluation. Differences:
* The loss and the prediction depend on `output_type` (see above) and the metric is the common-sense error in minutes instead of the accuracy.
* The best epoch is the one with the lowest **validation error** (the metric we want to minimise; the losses of different formulations cannot be compared).
* The test set is only evaluated when `evaluate_test = True`, so we do not pick hyperparameters with it.

In [ ]:
def make_optimizer(model, name, learning_rate, weight_decay = 0.0, momentum = 0.9):
    if name == "SGD":
        return optim.SGD(model.parameters(), lr = learning_rate, momentum = momentum, weight_decay = weight_decay)
    if name == "Adam":
        return optim.Adam(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
    if name == "AdamW":
        return optim.AdamW(model.parameters(), lr = learning_rate, weight_decay = weight_decay)
    raise ValueError(f"Unknown optimizer: {name!r}")

@torch.no_grad()
def evaluate(model, loader, params):
    """Average loss, mean common-sense error, and the per-image predicted / true times (in minutes)."""
    model.eval()
    loss_total, n = 0.0, 0
    preds, trues = [], []
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss_total += compute_loss(outputs, labels, params).item() * labels.size(0)
        n += labels.size(0)
        preds.append(outputs_to_minutes(outputs, params).cpu())
        trues.append(time_in_minutes(labels).float().cpu())
    preds, trues = torch.cat(preds), torch.cat(trues)
    mean_error = common_sense_error(preds, trues).mean().item()
    return loss_total / n, mean_error, preds, trues

def region_errors(pred, true, window = 30):
    """Mean error for times within `window` minutes of 12:00 (where the 12 -> 0 wrap-around is) and for all other times."""
    error = common_sense_error(pred, true)
    near = (true < window) | (true > 720 - window)
    return error[near].mean().item(), error[~near].mean().item()

def plot_history(history):
    epochs_range = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 4))

    # Loss Plot
    ax1.plot(epochs_range, history["train_loss"], marker = "o", label = "Train Loss")
    ax1.plot(epochs_range, history["val_loss"], marker = "o", label = "Validation Loss")
    ax1.set_title("Training loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True)

    # Error Plot
    ax2.plot(epochs_range, history["train_err"], marker = "o", label = "Train Error")
    ax2.plot(epochs_range, history["val_err"], marker = "o", label = "Validation Error")
    ax2.set_title("Common-sense error")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Error (minutes)")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

In [ ]:
MODEL_KEYS = ("output_type", "n_classes", "minute_mode", "cnn_channels", "kernel_size",
              "batch_norm", "hidden_dims", "activation", "dropout")

def run_experiment(params, train = None, validation = None, test = None,
                   plotting = False, evaluate_test = False, verbose = False):
    # By default we use the 75x75 datasets
    train = clock_train if train is None else train
    validation = clock_validation if validation is None else validation
    test = clock_test if test is None else test

    set_seed(params["seed"])
    if params["output_type"] == "classification":
        assert 720 % params["n_classes"] == 0, "n_classes must divide 720 (24, 72, 144, 720, ...)"
    start_time = time.time()

    # Create DataLoaders
    train_loader = DataLoader(train, batch_size = params["batch_size"], shuffle = True)
    validation_loader = DataLoader(validation, batch_size = params["batch_size"], shuffle = False)
    test_loader = DataLoader(test, batch_size = params["batch_size"], shuffle = False)

    # Create the model and the optimizer (only the keys the model needs are passed on)
    model = ClockNet(**{key: params[key] for key in MODEL_KEYS}).to(device)
    optimizer = make_optimizer(model, params["optimizer"], params["learning_rate"], params["weight_decay"])

    # Learning rate schedule: cosine decay from the learning rate to 0 over all epochs
    scheduler = None
    if params["scheduler"] == "cosine":
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = params["epochs"])

    # Store losses and errors for each epoch
    history = {"train_loss": [], "val_loss": [], "train_err": [], "val_err": []}
    best_val_loss, best_val_err, best_epoch, best_model_state = np.inf, np.inf, 0, None

    for epoch in range(params["epochs"]):
        #---------- Training Phase ----------
        model.train()
        loss_total, err_total, n = 0.0, 0.0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = compute_loss(outputs, labels, params)
            loss.backward()
            optimizer.step()

            # Training performance (a running average over the epoch, measured in train mode)
            loss_total += loss.item() * labels.size(0)
            with torch.no_grad():
                err = common_sense_error(outputs_to_minutes(outputs, params), time_in_minutes(labels).float())
            err_total += err.sum().item()
            n += labels.size(0)

        history["train_loss"].append(loss_total / n)
        history["train_err"].append(err_total / n)

        #---------- Validation Phase ----------
        val_loss, val_err, _, _ = evaluate(model, validation_loader, params)
        history["val_loss"].append(val_loss)
        history["val_err"].append(val_err)

        if scheduler is not None:
            scheduler.step()

        if verbose:
            print(f"Epoch {epoch + 1:>3}/{params['epochs']} | train loss {history['train_loss'][-1]:.4f} | "
                  f"val loss {val_loss:.4f} | train err {history['train_err'][-1]:.1f} | val err {val_err:.1f} min")

        # Save the model with the lowest validation error (a copy of the weights on the CPU)
        if val_err < best_val_err:
            best_val_loss, best_val_err, best_epoch = val_loss, val_err, epoch + 1
            best_model_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

        # Stop if training diverged (NaN / inf)
        if not np.isfinite(val_loss):
            break

    # Get the best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    history["best_epoch"] = best_epoch

    # Per-image predictions of the best model on the validation set (used for the error analysis)
    _, _, history["val_pred"], history["val_true"] = evaluate(model, validation_loader, params)

    #---------- Test Phase (only on request) ----------
    test_loss, test_err = None, None
    if evaluate_test:
        test_loss, test_err, history["test_pred"], history["test_true"] = evaluate(model, test_loader, params)

    print(f"  done in {time.time() - start_time:.0f}s | best validation error {best_val_err:.2f} min (epoch {best_epoch})")
    if plotting:
        plot_history(history)

    return model, history, best_val_loss, best_val_err, test_loss, test_err

In [ ]:
def run_grid(base_params, grid):
    """Run every combination of the values in `grid` ({parameter name: list of values}) on top of `base_params`.
    Returns a results table (best first) and the parameters of the best run."""
    rows, best_err, best_params = [], np.inf, None

    for values in itertools.product(*grid.values()):
        changes = dict(zip(grid.keys(), values))
        print("Testing ->", changes)
        params = {**base_params, **changes}

        model, history, val_loss, val_err, _, _ = run_experiment(params)
        near, elsewhere = region_errors(history["val_pred"], history["val_true"])
        rows.append({
            "Formulation": params["output_type"],
            "Setting": ", ".join(f"{key}={value}" for key, value in changes.items()),
            "Val Err (min)": round(val_err, 2),
            "Err near 12:00": round(near, 2),
            "Err elsewhere": round(elsewhere, 2),
            "Best epoch": history["best_epoch"],
            "Params": sum(p.numel() for p in model.parameters()),
        })
        if best_params is None or val_err < best_err:
            best_err, best_params = val_err, params

    results = pd.DataFrame(rows).sort_values("Val Err (min)").reset_index(drop = True)
    return results, best_params

## Experiments on the 75x75 images (validation set)

Parameters

* Output: `output_type` ("classification", "regression", "multihead", "trig"), `n_classes`, `minute_mode`, `alpha`, `loss` ("mse", "l1", "smoothl1")
* Architecture: `cnn_channels`, `kernel_size`, `batch_norm`, `hidden_dims`, `activation`, `dropout`
* Optimizer: `optimizer`, `learning_rate`, `weight_decay`, `scheduler`
* Training: `epochs`, `batch_size`, `seed`

First a baseline (regression with the L1 loss) with training curves.

In [ ]:
base_params = {
    # Output formulation
    "output_type": "regression",    # "classification", "regression", "multihead" or "trig"
    "n_classes": 24,                # only for classification (must divide 720)
    "minute_mode": "class",         # only for multihead: "class" (60 classes) or "reg" (one value)
    "alpha": 1.0,                   # only for multihead: L = L_hour + alpha * L_minute
    "loss": "l1",                   # for real-valued outputs: "mse", "l1" or "smoothl1"

    # Architecture
    "cnn_channels": [32, 64, 128, 256],
    "kernel_size": 3,
    "batch_norm": True,
    "hidden_dims": [256],
    "activation": nn.ReLU,
    "dropout": 0.0,

    # Optimizer
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "weight_decay": 0.0,
    "scheduler": "cosine",          # "cosine" or None

    # Training
    "epochs": 20,
    "batch_size": 64,
    "seed": 67
}

model0, history0, val_loss0, val_err0, _, _ = run_experiment(base_params, plotting = True, verbose = True)

print("\nBaseline results:")
print(f"Best Validation Error: {val_err0:.2f} minutes (epoch {history0['best_epoch']}) | Parameters: {sum(p.numel() for p in model0.parameters()):,}")

### 1. Classification

Bins of 30 / 10 / 5 / 1 minutes (24 / 72 / 144 / 720 classes). The predicted time is the centre of the predicted bin.

*For the report:* what happens to the error when the number of classes grows? Think about the number of training images per class (14,400 / 720 = 20), about neighbouring classes being treated as completely unrelated by the cross-entropy loss, and about the size of the output layer.

In [ ]:
grid_cls = {"n_classes": [24, 72, 144, 720]}
results_cls, best_cls = run_grid({**base_params, "output_type": "classification"}, grid_cls)
display(results_cls)

### 2. Regression

One output (time as a fraction of 12 hours). Which loss is best? Look at the columns **Err near 12:00** (true time within 30 minutes of 12:00) and **Err elsewhere**.

*For the report:* 11:59 and 0:01 are 2 minutes apart, but as numbers they are almost the whole range apart (0.998 vs 0.001). The loss cannot know that the clock is circular, and the images around 12:00 look almost identical, so the network is pulled towards both ends.

In [ ]:
grid_reg = {"loss": ["mse", "l1", "smoothl1"]}
results_reg, best_reg = run_grid({**base_params, "output_type": "regression"}, grid_reg)
display(results_reg)

### 3. Multi-head

Shared backbone, two heads: `out_hour, out_minute = model(x)`. The hour head always classifies the 12 hours. The minute head either classifies the 60 minutes (`"class"`) or regresses `minute / 60` (`"reg"`). Joint loss: L = L_hour + alpha x L_minute. The cross-entropy and the regression loss have a different scale, so alpha is explored for both.

*For the report:* compare with the single-output models above. Which mistakes are typical (hint: around the full hour, when the hour hand is almost on a number and the hour and minute heads can disagree)?

In [ ]:
grid_multi = {"minute_mode": ["class", "reg"], "alpha": [0.5, 1.0, 5.0]}
results_multi, best_multi = run_grid({**base_params, "output_type": "multihead"}, grid_multi)
display(results_multi)

### 4. Trigonometric encoding (optional)

The hands are angles, so we predict the points on the unit circle (sin, cos) of the hour-hand angle and of the minute-hand angle. Design of the output layer (see `ClockNet`):
* 4 linear outputs, no ReLU / sigmoid (sin and cos are in [-1, 1] and can be negative).
* Each (sin, cos) pair is scaled to length 1, so the output is a valid point on the circle. Loss: MSE or L1 against the true (sin, cos).
* Decoding: `atan2` gives the two angles. The minute comes from the minute hand (exact), the hour from the hour hand (rough): we choose the hour for which the time fits the hour hand best. Because 11:59 and 0:01 are neighbours on the circle, there is no discontinuity at 12:00.

In [ ]:
grid_trig = {"loss": ["mse", "l1"]}
results_trig, best_trig = run_grid({**base_params, "output_type": "trig"}, grid_trig)
display(results_trig)

### Comparison of the formulations (validation set)

The best run of every formulation, and the winner.

In [ ]:
best_configs = {"classification": best_cls, "regression": best_reg, "multihead": best_multi, "trig": best_trig}

all_results = pd.concat([results_cls, results_reg, results_multi, results_trig])
all_results = all_results.sort_values("Val Err (min)").reset_index(drop = True)
display(all_results)

summary = all_results.drop_duplicates("Formulation").reset_index(drop = True)    # best run per formulation
display(summary)

winner = summary.loc[0, "Formulation"]
print("Best formulation on the validation set:", winner)

### Backbone tuning

Learning rate, width and dropout for the winning formulation (a small grid; add `weight_decay`, `kernel_size`, ... if you have time).

In [ ]:
grid_backbone = {
    "learning_rate": [1e-3, 3e-4],
    "cnn_channels": [[32, 64, 128, 256], [64, 128, 256, 512]],
    "dropout": [0.0, 0.3]
}
results_backbone, best_backbone = run_grid(best_configs[winner], grid_backbone)
display(results_backbone)

## Final results on the 150x150 images (test set)

All four formulations are trained on the 150x150 images with their best settings from above and the tuned backbone. We select the final model with the **validation** error; the test error is only reported.

In [ ]:
final_epochs = 30
clock150_train, clock150_validation, clock150_test = load_clock_data(150)

tuned = {key: best_backbone[key] for key in ("learning_rate", "cnn_channels", "dropout")}
final_rows, final_runs = [], {}

for name, config in best_configs.items():
    print(f"Final training -> {name}")
    params = {**config, **tuned, "epochs": final_epochs}
    model, history, val_loss, val_err, test_loss, test_err = run_experiment(
        params, clock150_train, clock150_validation, clock150_test, evaluate_test = True, verbose = True)

    near, elsewhere = region_errors(history["test_pred"], history["test_true"])
    final_rows.append({
        "Formulation": name,
        "Val Err (min)": round(val_err, 2),
        "Test Err (min)": round(test_err, 2),
        "Test near 12:00": round(near, 2),
        "Test elsewhere": round(elsewhere, 2)
    })
    final_runs[name] = (model, history)

final_table = pd.DataFrame(final_rows).sort_values("Val Err (min)").reset_index(drop = True)
display(final_table)

### Error analysis of the final model

In [ ]:
best_name = final_table.loc[0, "Formulation"]
final_model, final_history = final_runs[best_name]
errors = common_sense_error(final_history["test_pred"], final_history["test_true"])

print(f"Final model: {best_name}")
print(f"Test error: mean {errors.mean():.2f} | median {errors.median():.2f} | 90th percentile {errors.quantile(0.9):.2f} minutes")
for tolerance in (1, 5, 10, 30):
    print(f"Within {tolerance:>2} minutes: {100 * (errors <= tolerance).float().mean():.1f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 4))
ax1.hist(errors.numpy(), bins = 50)
ax1.set_title("Distribution of the test errors")
ax1.set_xlabel("Common-sense error (minutes)")
ax1.set_ylabel("Number of test images")

ax2.scatter(final_history["test_true"].numpy() / 60, errors.numpy(), s = 5)
ax2.set_title("Error versus true time")
ax2.set_xlabel("True time (hours since 12:00)")
ax2.set_ylabel("Common-sense error (minutes)")
plt.tight_layout()
plt.show()

In [ ]:
def clock_string(minutes):
    minutes = int(round(float(minutes))) % 720
    return f"{minutes // 60}:{minutes % 60:02d}"

# Some test images with the true and the predicted time
fig, ax = plt.subplots(3, 3, figsize = (9, 9))
for i, axis in enumerate(ax.flatten()):
    image, _ = clock150_test[i]
    axis.imshow(image.squeeze(), cmap = "gray")
    axis.set_title(f"true {clock_string(final_history['test_true'][i])} | pred {clock_string(final_history['test_pred'][i])} "
                   f"| error {errors[i]:.0f} min", fontsize = 8)
    axis.axis("off")
plt.tight_layout()
plt.show()

torch.save(final_model.state_dict(), "clock_model.pt")